In [ ]:
%env CUDA_VISIBLE_DEVICES=1
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import os
from ne_aligned_utils.plot import plot_aligned_embedding
from ne_aligned_utils.utils import sparsify_plan
from ne_aligned_utils.data import make_fig1_data
from ne_aligned_utils.utils import get_path

In [ ]:
root_path = get_path("data")
fig_path = os.path.join(root_path, "figures")

In [ ]:
dataset = "fig1"

tol = 1e-12
maxIter = int(1e4)
res_path = os.path.join("data", dataset)  # adjust as needed, or rather abstract graph as in other repos
fig_path = os.path.join("results", "figures")


n0 = 1000
n1 = 5000
colors =  ["r"]*n0 + ["b"]*n1
seeds = [0, 1, 2, 3, 4]
eps_times_list = [1, 10, 100]
embd_dim = 2




In [ ]:

# show data

x0, x1 = make_fig1_data(seed=0)
# plot in 3d
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*x0.T, c='r', s=1)
ax.scatter(*x1.T, c='b', s=1)
ax.set_box_aspect([1,1,1])  # Equal aspect ratio

fig.suptitle("Original data")
fig.savefig(os.path.join(fig_path, "fig1_data.png"))

In [ ]:
# show plans
sparsify = None # "topk" # None # "topk" # None

seed = 0
t=0
sparsity = "topk"

fig, ax = plt.subplots( len(eps_times_list), 2, figsize=(10, 5))

for i, eps_times in enumerate(eps_times_list):

    filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}.npz")
    a = np.load(os.path.join(res_path, filename))
    P = a["P_sparse"]

    P_topk = sparsify_plan(P, mode=sparsity, k=15)

    ax[i, 0].imshow(P, interpolation="none")
    ax[i, 0].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.15, y=0.5, transform=ax[i, 0].transAxes)

    ax[i, 1].imshow(P_topk)

    if i == 0:
        ax[i, 0].set_title("Full plan")
        ax[i, 1].set_title("Top15 sparsification")

fig.suptitle("EOT plans for different regularization and sparsification")
fig.savefig(os.path.join(fig_path, "fig1_plans.png"))

In [ ]:
# colorbar is bad, plan with normal eps is non-sparse
eps_times = 1
sparsify = None # "topk" # None # "topk" # None

seed = 0
t=0
filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}.npz")
a = np.load(os.path.join(res_path, filename))
P = a["P_sparse"]

plt.imshow(P.T, interpolation="none")
plt.colorbar()

In [ ]:
# plan with low eps is effectively sparse
eps_times = 100
sparsify = None # "topk" # None # "topk" # None

seed = 0
t=0
filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}.npz")
a = np.load(os.path.join(res_path, filename))
P = a["P_sparse"]

plt.imshow(P.T, interpolation="none")
plt.colorbar()

In [ ]:
# common hyperparams for NE runs
init_str = "random"  # "pca" or "random"
init_scale = 1

batch_size = 2**14
negative_samples = 5
n_epochs = 200
loss_mode = "infonce"

In [ ]:
## CNE experiemnts

In [ ]:
# sample non-sparsified
alignment = True
sample_pos = "per_point"
sparsify = None



In [ ]:

fig, ax = plt.subplots(len(eps_times_list),len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                            f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                            f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               s=0.1,
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("No sparsification, pos. samples per point, with alignment loss")
fig.savefig(os.path.join(fig_path, "fig1_per_point_sampling.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                                f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                                f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
# iterate over sparsified

alignment = True
sample_pos = None #"per_point"
sparsify = "topk"


In [ ]:
fig, ax = plt.subplots( len(eps_times_list), len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                            f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                            f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               s=0.1,
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("Top15 sparsification, pos. samples over whole plan, with alignment loss")
fig.savefig(os.path.join(fig_path, "fig1_topk_alignment.png"))


In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                                f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                                f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
## no alignement loss
alignment = False
sample_pos = None #"per_point"
sparsify = "topk"


In [ ]:
fig, ax = plt.subplots( len(eps_times_list), len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                            f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                            f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               s=0.1,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)
fig.suptitle("Top15 sparsification, pos. samples over whole plan, no alignment loss")
fig.savefig(os.path.join(fig_path, "fig1_topk_no_alignment.png"))


In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                                f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                                f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
# EOT

In [ ]:
sparsify = None # "topk" # None # "topk" # None


In [ ]:
t=0

fig, ax = plt.subplots( len(eps_times_list), len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               s=0.1,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)
fig.suptitle("EOT0 embeddings")
fig.savefig(os.path.join(fig_path, "fig1_EOT0.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
t=1

fig, ax = plt.subplots( len(eps_times_list), len(seeds), figsize=(6, 4.5))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               s=0.1,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("EOT1 embeddings")
fig.savefig(os.path.join(fig_path, "fig1_EOT1.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
# t-SNE

In [ ]:
# common hyperparams for NE runs
init_str = "random"  # "pca" or "random"
init_scale = 10000

n_epochs = 200
norm = "whole"

In [ ]:
# sample topk
sparsify = "topk"

In [ ]:

fig, ax = plt.subplots(len(eps_times_list),len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               s=0.1,
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("Topk sparsification tSNE")
fig.savefig(os.path.join(fig_path, "fig1_topk_tsne.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
# sample topk
sparsify = "weighted_topk"

In [ ]:

fig, ax = plt.subplots(len(eps_times_list),len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               s=0.1,
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("Weighted topk sparsification tSNE")
fig.savefig(os.path.join(fig_path, "fig1_weighted_topk_tsne.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
# sample None
sparsify = None

In [ ]:

fig, ax = plt.subplots(len(eps_times_list),len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               s=0.1,
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("No sparsification tSNE")
fig.savefig(os.path.join(fig_path, "fig1_non_sparse_tsne.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
## norm per point

In [ ]:
# common hyperparams for NE runs
init_str = "random"  # "pca" or "random"
init_scale = 10000

n_epochs = 200
norm = "per_point"

In [ ]:
# sample topk
sparsify = "topk"

In [ ]:

fig, ax = plt.subplots(len(eps_times_list),len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               s=0.1,
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("Topk sparsification, norm per point tSNE")
fig.savefig(os.path.join(fig_path, "fig1_topk_per_point_tsne.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
# sample None
sparsify = None

In [ ]:

fig, ax = plt.subplots(len(eps_times_list),len(seeds), figsize=(6, 4))

for j, seed in enumerate(seeds):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               s=0.1,
                               )
        if i == 0:
            ax[i, j].set_title(f"seed {seed}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle("No sparsification norm per point tSNE")
fig.savefig(os.path.join(fig_path, "fig1_non_sparse_per_point_tsne.png"))

In [ ]:
for eps_times in eps_times_list:
    metrics_path = os.path.join(res_path,
                                f"{dataset}_epstimes_{eps_times}_sparsify_{sparsify}_norm_{norm}_dim_{embd_dim}_init_{init_str}" + \
                                f"_scale_{init_scale}_epochs_{n_epochs}_tsne_metrics.npz")
    metrics = np.load(metrics_path)
    print(f"eps times: {eps_times}")
    print(f"Concordance: {metrics['concordance'].mean():.3f} ± {metrics['concordance'].std():.3f}")
    print(f"Cross concordance: {metrics['cross_concordance'].mean():.3f} ± {metrics['cross_concordance'].std():.3f}")
    print()

In [ ]:
## check out parallel lines

# iterate over sparsified

alignment = True
sample_pos = None  #"per_point"
sparsify = "topk"

init_scale = 1

seed = 1
eps_times = 1
filename = os.path.join(res_path,
                        f"{dataset}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}" +
                        f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}" +
                        f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}.npz")
a = np.load(os.path.join(res_path, filename))
embd = a["embd"]
P_sparse = a["P_sparse"]


In [ ]:
data0, data1 = make_fig1_data(seed=seed)


In [ ]:
colors =  ["r"]*n0 + ["b"]*n1

fig, ax = plt.subplots(1, 3, figsize=(8, 5))

plot_aligned_embedding(embd[::-1],
                       n0,
                       color=colors[::-1],
                       shuffle=False,
                       ax=ax[0],
                       s=1,
                       )
ax[0].set_title("Original dataset")


cmap = plt.get_cmap("viridis")
colors_angle = [cmap(i / n0) for i in range(n0)] + [cmap(i / n1) for i in range(n1)]

plot_aligned_embedding(embd[::-1],
                       n0,
                       color=colors_angle[::-1],
                       shuffle=False,
                       ax=ax[1],
                       s=1,
                       )

ax[1].set_title("Angle")

clean0, clean1 = make_fig1_data(seed=seed, clean=True)

clean = np.concatenate((clean0, clean1))

radii = np.sqrt(np.sum(clean ** 2, axis=1))
radii -= radii.min()
radii /= radii.max()
color_radius = [cmap(r) for r in radii]

plot_aligned_embedding(embd[::-1],
                       n0,
                       color=color_radius[::-1],
                       shuffle=False,
                       ax=ax[2],
                       s=0.5,
                       )

ax[2].set_title("Clean Radius")

fig.savefig(os.path.join(fig_path, "fig1_parallel_line.png"), dpi=300)

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
nbrs = NearestNeighbors(n_neighbors=15, algorithm='ball_tree').fit(embd)

_, idx = nbrs.kneighbors(embd)


In [ ]:
mask0 = (idx[:n0] >= n0).sum(1) > 15/2
mask1 = (idx[n0:] <= n0).sum(1) > 15/2

In [ ]:
mask0

In [ ]:
cmap = plt.get_cmap("viridis")
color_outliers0 = [cmap(i) for i in mask0]

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(6, 4))


ax[0].scatter(*embd[:n0][~mask0].T, s=1, c="k")
ax[0].scatter(*embd[:n0][mask0].T, s=1,c="g")
ax[0].set_aspect("equal")
ax[0].axis("off")
ax[0].set_title("Ring")

ax[1].scatter(*embd[n0:][~mask1].T, s=1, c="k", label="Normal")
ax[1].scatter(*embd[n0:][mask1].T, s=1,c="g", label="Outliers")
ax[1].set_aspect("equal")
ax[1].axis("off")
ax[1].set_title("Cylinder")

ax[1].legend(loc="lower right")
fig.savefig(os.path.join(fig_path, "outlier_locations.png"), dpi=300)

In [ ]:
# show data

x0, x1 = make_fig1_data(seed=seed)
# plot in 3d
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*x0[~mask0].T, c='k', s=1)
ax.scatter(*x0[mask0].T, c='g', s=1)
ax.scatter(*x1[~mask1].T, c='k', s=1)
ax.scatter(*x1[mask1].T, c='g', s=1)
ax.set_box_aspect([1,1,1])  # Equal aspect ratio

fig.suptitle("Original data")
fig.savefig(os.path.join(fig_path, "fig1_data_odd.png"))

In [ ]:
# show data

x0, x1 = make_fig1_data(seed=seed)
x0 = x0[:, :2]
x1 = x1[:, :2]
# plot in 3d
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(*x0[~mask0].T, c='k', s=1)
ax.scatter(*x0[mask0].T, c='g', s=1)
ax.scatter(*x1[~mask1].T, c='k', s=1)
ax.scatter(*x1[mask1].T, c='g', s=1)
ax.set_aspect("equal")
ax.axis("off")
fig.savefig(os.path.join(fig_path, "fig1_data_odd_top.png"))

In [ ]:
mass0 = P_sparse.sum(1)
mass1 = P_sparse.sum(0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 4))


ax[0].boxplot([mass0[mask0], mass0[~mask0]], tick_labels=["Outliers Ring", "Normal Ring"])
ax[0].set_ylabel("Mass")

ax[1].boxplot([mass1[mask1], mass1[~mask1]], tick_labels=["Outliers Cylinder", "Normal Cylinder"])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].scatter(np.arange(n0)[~mask0], mass0[~mask0], s=5, c="k")
ax[0].scatter(np.arange(n0)[mask0],  mass0[mask0],  s=5, c="g")
ax[0].set_ylim(0, None)
ax[0].set_ylabel("Mass")
ax[0].set_title("Ring")

ax[1].scatter(np.arange(n1)[~mask1], mass1[~mask1],label="Normal", s=5, c="k")
ax[1].scatter(np.arange(n1)[mask1],  mass1[mask1], label="Outlier", s=5, c="g")
ax[1].set_ylim(0, None)
ax[1].legend()
ax[1].set_title("Cylinder")

fig.savefig(os.path.join(fig_path, "outlier_mass.png"), dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
ax.imshow(P_sparse, interpolation="none")
fig.savefig(os.path.join(fig_path, "fig1_eps1_topk.png"), dpi=600)